# Azure VM デプロイ手順書 — Legacy Bookstore Application

このノートブックでは、Struts 1.x / Hibernate 3.6 / MySQL のレガシー Java アプリを  
**Azure VM (Ubuntu 22.04 LTS, Standard_B2s, Japan East)** に立ち上げ、動作確認するまでの  
手順をステップバイステップで実行します。

## アーキテクチャ概要
```
[ローカル PC]                [Azure Japan East]
  ant clean build  ──scp──►  Ubuntu 22.04 VM (Standard_B2s)
  config/mysql/     ──ssh──►    ├── Tomcat 9 (port 8080)
  SQL scripts                   ├── OpenJDK 11
                                ├── MySQL 8.0 (127.0.0.1:3306)
                                └── /etc/hosts: legacy-mysql → 127.0.0.1
```

## 前提条件
| ツール | バージョン | 用途 |
|--------|-----------|------|
| Azure CLI | 2.50+ | Azure リソース操作 |
| Apache Ant | 1.9+ | WAR ビルド |
| JDK | 8 or 11 | ローカルビルド用 |
| OpenSSH | - | SCP / SSH (Windows 10+ 標準) |

## 実行順序
1. 設定 → 2. ログイン → 3. Bicep バリデーション → 4. デプロイ  
→ 5. cloud-init 完了待ち → 6. WAR ビルド → 7. WAR アップロード  
→ 8. DB 初期化 → 9. 動作確認 → (10. クリーンアップ)

## ⚙️ Step 0: 設定変数
**このセルを最初に編集してください。** サブスクリプション ID とパスワードは必ず変更してください。

In [ ]:
import os, json, subprocess, pathlib, time

# ─────────────────────────────────────────────────────────────────────────────
# ★ ここを編集してください
# ─────────────────────────────────────────────────────────────────────────────
SUBSCRIPTION_ID  = "YOUR_SUBSCRIPTION_ID"          # Azure サブスクリプション ID
ADMIN_PASSWORD   = "P@ssw0rd-LegacyDemo2024!"      # ★ 必ず変更する (大文字・小文字・数字・記号を含む8文字以上)

RESOURCE_GROUP   = "rg-legacy-bookstore"
LOCATION         = "japaneast"
VM_NAME          = "vm-legacy-bookstore"
ADMIN_USERNAME   = "azureuser"
DEPLOYMENT_NAME  = "deploy-legacy-bookstore"
# ─────────────────────────────────────────────────────────────────────────────

# パス自動設定 (notebooks/ の親がワークスペースルート)
NOTEBOOK_DIR   = pathlib.Path(os.getcwd())
WORKSPACE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

BICEP_FILE     = WORKSPACE_ROOT / "infra" / "bicep" / "main.bicep"
PARAMS_FILE    = WORKSPACE_ROOT / "infra" / "bicep" / "parameters" / "dev.bicepparam"
SQL_DIR        = WORKSPACE_ROOT / "config" / "mysql"
DIST_DIR       = WORKSPACE_ROOT / "dist"

print("=== 設定確認 ===")
print(f"Subscription : {SUBSCRIPTION_ID}")
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Location      : {LOCATION}")
print(f"VM Name       : {VM_NAME}")
print(f"Admin User    : {ADMIN_USERNAME}")
print(f"BICEP_FILE    : {BICEP_FILE}")
print(f"WORKSPACE_ROOT: {WORKSPACE_ROOT}")


In [ ]:
import base64

# cloud-init.yaml を base64 エンコード (Bicep の customData param として渡す)
CLOUD_INIT_FILE = WORKSPACE_ROOT / "infra" / "scripts" / "cloud-init.yaml"
CUSTOM_DATA_B64 = base64.b64encode(CLOUD_INIT_FILE.read_bytes()).decode("ascii")

print(f"✅ cloud-init エンコード完了: {len(CUSTOM_DATA_B64)} 文字 (base64)")


---
## 🔑 Step 1: Azure ログイン

まだログインしていない場合はブラウザが開きます。  
ログイン済みの場合はスキップできます。

In [ ]:
# Azure CLI ログイン状態を確認し、未ログインなら az login を実行
result = subprocess.run(
    ["az", "account", "show", "--output", "json"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("未ログインです。ブラウザが開きます...")
    subprocess.run(["az", "login"], check=True)
else:
    account = json.loads(result.stdout)
    print(f"✅ ログイン済み: {account['user']['name']}")
    print(f"   テナント: {account['tenantId']}")

# 対象サブスクリプションを設定
subprocess.run(
    ["az", "account", "set", "--subscription", SUBSCRIPTION_ID],
    check=True
)

# 確認
result2 = subprocess.run(
    ["az", "account", "show", "--output", "json"],
    capture_output=True, text=True, check=True
)
current = json.loads(result2.stdout)
print(f"\n✅ アクティブサブスクリプション: {current['name']} ({current['id']})")


---
## 🔍 Step 2: Bicep テンプレートのバリデーション

デプロイ前に Bicep テンプレートの構文チェックと what-if 検証を行います。

In [ ]:
# Bicep CLI がインストールされていない場合はインストール
subprocess.run(["az", "bicep", "install"], check=True)
print("✅ Bicep CLI ready")

# Bicep ビルド検証 (ARM JSON へのコンパイル確認)
result = subprocess.run(
    ["az", "bicep", "build", "--file", str(BICEP_FILE)],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("❌ Bicep ビルドエラー:")
    print(result.stderr)
else:
    print("✅ Bicep ビルド成功")

# What-if: 実際に作成されるリソースを事前確認
print("\n--- What-if 実行中 (変更内容のプレビュー) ---")
result_wf = subprocess.run(
    [
        "az", "deployment", "sub", "what-if",
        "--location", LOCATION,
        "--template-file", str(BICEP_FILE),
        "--parameters", str(PARAMS_FILE),
        "--parameters", f"adminPassword={ADMIN_PASSWORD}",
        "--parameters", f"customData={CUSTOM_DATA_B64}",
        "--name", DEPLOYMENT_NAME,
    ],
    capture_output=True, text=True
)
print(result_wf.stdout)
if result_wf.returncode != 0:
    print("⚠️ What-if エラー (無視して続行可):", result_wf.stderr[:500])


---
## 🚀 Step 3: インフラのデプロイ

Bicep テンプレートを Azure にデプロイします。  
VM 作成・cloud-init 実行まで含めて **約 3～5 分** かかります。

In [ ]:
print("デプロイ開始... (Ctrl+C で中断可)")
deploy_result = subprocess.run(
    [
        "az", "deployment", "sub", "create",
        "--location", LOCATION,
        "--template-file", str(BICEP_FILE),
        "--parameters", str(PARAMS_FILE),
        "--parameters", f"adminPassword={ADMIN_PASSWORD}",
        "--parameters", f"customData={CUSTOM_DATA_B64}",
        "--name", DEPLOYMENT_NAME,
        "--output", "json",
    ],
    capture_output=True, text=True
)

if deploy_result.returncode != 0:
    print("❌ デプロイ失敗:")
    print(deploy_result.stderr)
    raise RuntimeError("デプロイに失敗しました。エラー内容を確認してください。")

deployment = json.loads(deploy_result.stdout)
print("✅ デプロイ成功!")
print(f"   状態: {deployment['properties']['provisioningState']}")
print(f"   デプロイ名: {deployment['name']}")


In [ ]:
# デプロイ出力から接続情報を取得
outputs = deployment["properties"]["outputs"]

PUBLIC_IP   = outputs["publicIpAddress"]["value"]
SSH_COMMAND = outputs["sshCommand"]["value"]
APP_URL     = outputs["appUrl"]["value"]

print("=" * 55)
print("🖥️  接続情報")
print("=" * 55)
print(f"Public IP   : {PUBLIC_IP}")
print(f"SSH コマンド: {SSH_COMMAND}")
print(f"アプリ URL  : {APP_URL}")
print("=" * 55)
print(f"\n📝 メモ: ADMIN_PASSWORD = {ADMIN_PASSWORD}")


---
## ⏳ Step 4: cloud-init 完了待ち

VM の初回起動時に cloud-init が自動実行され、以下をインストール・設定します。

- OpenJDK 11, Tomcat 9, MySQL 8.0, Ant
- MySQL データベース・ユーザー作成 (`legacy_db` / `legacy_user`)
- `/etc/hosts` に `legacy-mysql → 127.0.0.1` を追加

**完了まで約 5～10 分かかります。** 以下のセルで SSH 接続できることを確認します。

In [ ]:
# SSH 接続確認 (cloud-init 完了待ち)
# cloud-init が完了するまで SSH 接続が失敗することがあります
print(f"SSH 接続確認中: {PUBLIC_IP}")
print("(パスワード入力が必要です。ターミナルで直接実行することもできます)\n")

max_attempts = 20
for attempt in range(1, max_attempts + 1):
    chk = subprocess.run(
        [
            "ssh",
            "-o", "StrictHostKeyChecking=no",
            "-o", "ConnectTimeout=10",
            "-o", "BatchMode=yes",         # パスワード不要で確認するだけ
            f"{ADMIN_USERNAME}@{PUBLIC_IP}",
            "echo OK",
        ],
        capture_output=True, text=True
    )
    if chk.returncode == 0:
        print(f"✅ SSH 接続成功 (試行 {attempt}回目)")
        break
    print(f"   [{attempt:02d}/{max_attempts}] 待機中... (30秒後に再試行)")
    time.sleep(30)
else:
    print("⚠️ SSH 接続タイムアウト。手動で再試行してください:")
    print(f"   ssh {ADMIN_USERNAME}@{PUBLIC_IP}")


In [ ]:
# cloud-init の完了状態を確認 (SSH が成功した後に実行)
# パスワード認証の場合は下記コマンドをターミナルで直接実行してください
print("cloud-init 完了確認コマンド (ターミナルで実行):")
print(f"  ssh {ADMIN_USERNAME}@{PUBLIC_IP} 'sudo cloud-init status --wait'")
print()
print("または cloud-init ログを確認:")
print(f"  ssh {ADMIN_USERNAME}@{PUBLIC_IP} 'sudo tail -50 /var/log/cloud-init-output.log'")
print()
print("Tomcat 起動確認:")
print(f"  ssh {ADMIN_USERNAME}@{PUBLIC_IP} 'sudo systemctl status tomcat9 --no-pager'")
print()
print("MySQL 起動確認:")
print(f"  ssh {ADMIN_USERNAME}@{PUBLIC_IP} 'sudo systemctl status mysql --no-pager'")


---
## 🔨 Step 5: WAR ファイルのローカルビルド

ローカルで `ant clean build` を実行し、`dist/legacy-app.war` を生成します。

**前提条件 (ローカル PC)**:
- Apache Ant 1.9+（`ant -version` で確認）
- JDK 8 または 11（`java -version` で確認）
- `JAVA_HOME` 環境変数が設定済み

In [ ]:
# ant clean build 実行
print(f"作業ディレクトリ: {WORKSPACE_ROOT}")
build_result = subprocess.run(
    ["ant", "clean", "build"],
    capture_output=True, text=True,
    cwd=str(WORKSPACE_ROOT)
)

print(build_result.stdout[-3000:] if len(build_result.stdout) > 3000 else build_result.stdout)

if build_result.returncode != 0:
    print("❌ ビルド失敗:")
    print(build_result.stderr)
    print("\n💡 Ant がインストールされていない場合:")
    print("   Windows: https://ant.apache.org/bindownload.cgi からダウンロード")
    print("   または PATH に ant が含まれているか確認してください")
    raise RuntimeError("ビルドに失敗しました")

WAR_FILE = DIST_DIR / "legacy-app.war"
if WAR_FILE.exists():
    size_kb = WAR_FILE.stat().st_size // 1024
    print(f"✅ WAR ファイル生成: {WAR_FILE} ({size_kb} KB)")
else:
    print(f"❌ WAR ファイルが見つかりません: {WAR_FILE}")
    raise FileNotFoundError(f"WAR ファイルが見つかりません: {WAR_FILE}")


---
## 📤 Step 6: WAR ファイルのアップロードとデプロイ

WAR ファイルを SCP で VM に転送し、Tomcat の webapps に配置します。

> **⚠️ パスワード入力について**  
> SCP/SSH を実行すると `azureuser@<IP>'s password:` のプロンプトが表示されます。  
> Step 0 で設定した `ADMIN_PASSWORD` を入力してください。

In [ ]:
# 実行するコマンドを表示 (パスワード認証は対話的に行う必要があるためコマンドを生成)
SCP_CMD = (
    f"scp -o StrictHostKeyChecking=no "
    f"{WAR_FILE} "
    f"{ADMIN_USERNAME}@{PUBLIC_IP}:/tmp/legacy-app.war"
)
DEPLOY_CMD = (
    f"ssh -o StrictHostKeyChecking=no {ADMIN_USERNAME}@{PUBLIC_IP} "
    f"\"sudo cp /tmp/legacy-app.war /var/lib/tomcat9/webapps/ && "
    f"sudo systemctl restart tomcat9 && "
    f"echo 'Tomcat restarted'\""
)

print("以下のコマンドをターミナルで実行してください:\n")
print("# 1. WAR ファイルを VM に転送")
print(SCP_CMD)
print()
print("# 2. Tomcat webapps に配置して再起動")
print(DEPLOY_CMD)
print()
print("─" * 60)
print("または、Jupyter から直接実行 (パスワード入力プロンプトが表示されます):")


In [ ]:
# Jupyter から直接 SCP 実行 (パスワードプロンプトが表示されます)
import shlex
scp_args = [
    "scp",
    "-o", "StrictHostKeyChecking=no",
    str(WAR_FILE),
    f"{ADMIN_USERNAME}@{PUBLIC_IP}:/tmp/legacy-app.war",
]
print(f"実行: {' '.join(scp_args)}")
scp_result = subprocess.run(scp_args)  # 対話的パスワード入力のため capture_output=False

if scp_result.returncode == 0:
    print("✅ SCP 完了")
    
    # Tomcat に配置して再起動
    deploy_args = [
        "ssh",
        "-o", "StrictHostKeyChecking=no",
        f"{ADMIN_USERNAME}@{PUBLIC_IP}",
        "sudo cp /tmp/legacy-app.war /var/lib/tomcat9/webapps/ && sudo systemctl restart tomcat9 && echo 'Tomcat restarted OK'",
    ]
    print(f"実行: {' '.join(deploy_args)}")
    deploy_run = subprocess.run(deploy_args)
    if deploy_run.returncode == 0:
        print("✅ WAR デプロイ完了")
    else:
        print("❌ デプロイに失敗しました。上記のコマンドを手動で実行してください。")
else:
    print("❌ SCP に失敗しました。上記のコマンドをターミナルで手動実行してください。")


---
## 🗄️ Step 7: データベースの初期化

SQL スクリプト（テーブル作成 + シードデータ）を MySQL に流し込みます。

スクリプトは SSH 経由でパイプして実行します。

In [ ]:
sql_scripts = [
    SQL_DIR / "01-create-tables.sql",
    SQL_DIR / "02-seed-data.sql",
]

# 実行コマンドをまず表示
print("以下のコマンドをターミナルで実行してください (パスワード入力が 4 回発生します):\n")
for sql_file in sql_scripts:
    cmd = (
        f"ssh -o StrictHostKeyChecking=no {ADMIN_USERNAME}@{PUBLIC_IP} "
        f"\"mysql -u legacy_user -plegacy_pass legacy_db\" < {sql_file}"
    )
    print(f"# {sql_file.name}")
    print(cmd)
    print()

print("─" * 60)
print("または、Jupyter から直接実行:")


In [ ]:
for sql_file in sql_scripts:
    print(f"\n--- {sql_file.name} を実行中 ---")
    sql_content = sql_file.read_text(encoding="utf-8")
    
    # SSH 経由で mysql にパイプ (パスワード認証はプロンプト表示)
    ssh_proc = subprocess.run(
        [
            "ssh",
            "-o", "StrictHostKeyChecking=no",
            f"{ADMIN_USERNAME}@{PUBLIC_IP}",
            "mysql -u legacy_user -plegacy_pass legacy_db",
        ],
        input=sql_content,
        text=True,
        capture_output=True,
    )
    
    if ssh_proc.returncode == 0:
        print(f"✅ {sql_file.name} 完了")
    else:
        print(f"❌ {sql_file.name} 失敗:")
        print(ssh_proc.stderr[:500])
        print("\n⚠️ 手動実行してください (上記のコマンド参照)")


---
## ✅ Step 8: アプリケーションの動作確認

Tomcat が WAR を展開するまで **約 30 秒〜1 分** 待ちます。  
その後、HTTP アクセスでアプリケーションが応答することを確認します。

In [ ]:
import urllib.request, urllib.error

print(f"アプリ URL: {APP_URL}")
print("HTTP 応答確認中... (最大 3 分待機)\n")

for attempt in range(1, 19):
    try:
        req = urllib.request.Request(APP_URL, headers={"User-Agent": "notebook-check"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            status = resp.status
            content = resp.read(200).decode("utf-8", errors="replace")
            print(f"✅ HTTP {status} — アプリ応答確認 (試行 {attempt}回目)")
            print(f"   先頭 200 文字: {content[:200].strip()}")
            break
    except urllib.error.HTTPError as e:
        if e.code in (400, 404, 302, 301):
            # ステータスがエラーでも Tomcat は起動している
            print(f"✅ HTTP {e.code} — Tomcat 起動確認 (WAR 展開中の可能性)")
            break
        print(f"   [{attempt:02d}/18] HTTP {e.code} 待機中...")
    except Exception as e:
        print(f"   [{attempt:02d}/18] 接続失敗: {type(e).__name__} — 10 秒後に再試行")
    time.sleep(10)
else:
    print(f"⚠️ アプリへの接続に失敗。手動確認してください: {APP_URL}")

print()
print("=" * 60)
print("🎉 デプロイ完了! ブラウザでアクセスしてください:")
print(f"   {APP_URL}")
print()
print("ログインアカウント:")
print("   admin    / admin123")
print("   manager  / manager123")
print("   clerk    / clerk123")
print("=" * 60)


In [ ]:
# Tomcat ログの末尾を確認 (デバッグ用)
print("Tomcat ログ確認コマンド (ターミナルで実行):")
print(f"  ssh {ADMIN_USERNAME}@{PUBLIC_IP} 'sudo tail -100 /var/log/tomcat9/catalina.out'")
print()
print("SSH でアプリサーバーに接続:")
print(f"  {SSH_COMMAND}")


---
## 🧹 Step 9: クリーンアップ（任意）

ラボ終了後、課金を止めるためにリソースグループごと削除します。

> **⚠️ この操作は元に戻せません。**  
> VM・ネットワーク・パブリック IP など、リソースグループ内のすべてのリソースが削除されます。

In [ ]:
# ⚠️ クリーンアップ: リソースグループを削除
# 実行前に確認してください。この操作は元に戻せません。

CONFIRM_DELETE = False  # ★ True に変更してから実行

if CONFIRM_DELETE:
    print(f"🗑️  リソースグループ '{RESOURCE_GROUP}' を削除中...")
    del_result = subprocess.run(
        [
            "az", "group", "delete",
            "--name", RESOURCE_GROUP,
            "--yes",          # 確認プロンプトをスキップ
            "--no-wait",      # バックグラウンドで削除
        ],
        capture_output=True, text=True
    )
    if del_result.returncode == 0:
        print("✅ 削除開始 (--no-wait: バックグラウンドで削除が進行中)")
        print("   完了確認: az group show --name", RESOURCE_GROUP)
    else:
        print("❌ 削除失敗:", del_result.stderr)
else:
    print("⏸️  スキップ: CONFIRM_DELETE = False のため削除しません。")
    print("   削除する場合は CONFIRM_DELETE = True に変更してください。")
